# Apply available P11 cache to the misconception datasets

This notebook loads the reformatted MathDial train and test CSVs together, loads every currently available production P11 cache record for both splits, and applies usable annotations incrementally. Missing, invalid, unreadable or structurally mismatched cache records are reported and skipped; their dataset rows remain unchanged.

Each usable cache grid contributes:

- the five turn-level P/A/N labels;
- five `*_src` columns containing only thread identifiers such as `S1` or `S1|S2`; and
- the complete compact `threads` JSON on the dialogue's `solution` row only.

The reusable loading, validation, update and atomic-save logic lives in `extension/scripts/data_management/apply_cached_annotations.py`. This notebook keeps the experiment paths, write controls, execution and reporting visible to the reader. No API calls are made.

## 1. Setup and load both datasets

Both source CSVs are loaded before either split is updated. Blank strings remain blank rather than becoming NaN, which is important for incremental annotation. The cache directories are also created here so train and test have parallel production paths.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'data' / 'misconception').exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the repository.')
sys.path.insert(0, str(Path.cwd()))

from extension.scripts.data_management.apply_cached_annotations import (
    apply_available_cache,
    load_available_cache,
    save_in_place,
    split_summary,
)
from extension.scripts.data_management.load_annotation_data import load_dataset

MODEL_SLUG = 'moonshot-direct/kimi-k3-max'
MODEL_DIR = MODEL_SLUG.replace('/', '__').replace(':', '-')
PROMPT = 'P11'
DATA_DIR = Path('data/misconception')
CACHE_ROOT = Path('extension/artifacts/extraction_cache')

TRAIN_PATH = DATA_DIR / 'mathdial_train.csv'
TEST_PATH = DATA_DIR / 'mathdial_test.csv'
TRAIN_CACHE_DIR = CACHE_ROOT / 'train' / MODEL_DIR / PROMPT
TEST_CACHE_DIR = CACHE_ROOT / 'test' / MODEL_DIR / PROMPT
TRAIN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

train = load_dataset(TRAIN_PATH)
test = load_dataset(TEST_PATH)
dataset_summary = pd.DataFrame({
    'rows': {'train': len(train), 'test': len(test)},
    'dialogues': {
        'train': train['dialogue_id'].nunique(),
        'test': test['dialogue_id'].nunique(),
    },
})

print('repository:', Path.cwd())
print('train cache:', TRAIN_CACHE_DIR)
print('test cache:', TEST_CACHE_DIR)
display(dataset_summary)

## 2. Load all existing cache records

Only unsuffixed `{dialogue_id}.json` files are production inputs. `load_available_cache` separates usable records from records that are unreadable, invalid, or belong to the wrong split/model/prompt. Suffixed files are reported but ignored. This stage does not require complete cache coverage and does not modify either dataframe.

In [ ]:
train_cache, train_cache_audit, train_suffixed = load_available_cache(
    TRAIN_CACHE_DIR,
    split='train',
    model_slug=MODEL_SLUG,
    prompt=PROMPT,
)
test_cache, test_cache_audit, test_suffixed = load_available_cache(
    TEST_CACHE_DIR,
    split='test',
    model_slug=MODEL_SLUG,
    prompt=PROMPT,
)

cache_summary = pd.DataFrame({
    'production JSON files': {
        'train': len(train_cache_audit),
        'test': len(test_cache_audit),
    },
    'usable': {'train': len(train_cache), 'test': len(test_cache)},
    'invalid or unreadable': {
        'train': len(train_cache_audit) - len(train_cache),
        'test': len(test_cache_audit) - len(test_cache),
    },
    'ignored suffixed files': {
        'train': len(train_suffixed),
        'test': len(test_suffixed),
    },
})
display(cache_summary)

for split, audit, suffixed in [
    ('train', train_cache_audit, train_suffixed),
    ('test', test_cache_audit, test_suffixed),
]:
    problems = audit[audit['status'].ne('usable')] if not audit.empty else audit
    if not problems.empty:
        print(f'{split}: first invalid or unreadable records')
        display(problems.head(20))
    if suffixed:
        print(f'{split}: ignored suffixed files:', suffixed[:20])

## 3. How the shared application step works

`apply_available_cache` makes a dataframe copy and loops through the filtered records. Before changing a dialogue it checks the annotation ID, exact ordered unit list, P/A/N labels, unique `S#` thread IDs, and whether every source ID resolves to a thread in that record. A structural problem skips only that record; other available dialogues are still applied.

For an applied dialogue, all five labels and source columns are refreshed from the current cache. Multi-source lists are written as `S1|S2`. The thread list is serialized once on the solution row and cleared from that dialogue's other rows. Dialogues with no usable cache retain exactly what was loaded, which makes later runs incremental.

## 4. Update and optionally save the train dataset

`WRITE_TRAIN=False` performs the complete in-memory update and audit without changing `mathdial_train.csv`. Set it to `True` to atomically replace that file with the currently available annotations. Incomplete cache coverage does not block the write.

In [ ]:
WRITE_TRAIN = False

train_updated, train_apply_audit = apply_available_cache(train, train_cache)
display(split_summary(train, train_updated, train_apply_audit).to_frame('value'))
train_skipped = (
    train_apply_audit[train_apply_audit['status'].ne('applied')]
    if not train_apply_audit.empty else train_apply_audit
)
if not train_skipped.empty:
    print('First structurally skipped train cache records:')
    display(train_skipped.head(20))

if WRITE_TRAIN:
    train_write = save_in_place(train_updated, TRAIN_PATH)
    print(f"WRITTEN: {train_write['path']}")
    print(f"rows: {train_write['rows']:,}")
    print(f"dialogues: {train_write['dialogues']:,}")
    print(f"file size: {train_write['size_mib']:.2f} MiB")
else:
    print(f'NOT WRITTEN: WRITE_TRAIN=False for {TRAIN_PATH.name}')

## 5. Update and optionally save the test dataset

The test split is independent of train. Its cache is read from `extraction_cache/test/moonshot-direct__kimi-k3-max/P11/`. Set `WRITE_TEST=True` when you want to replace `mathdial_test.csv` with whatever usable test annotations are currently available.

In [ ]:
WRITE_TEST = False

test_updated, test_apply_audit = apply_available_cache(test, test_cache)
display(split_summary(test, test_updated, test_apply_audit).to_frame('value'))
test_skipped = (
    test_apply_audit[test_apply_audit['status'].ne('applied')]
    if not test_apply_audit.empty else test_apply_audit
)
if not test_skipped.empty:
    print('First structurally skipped test cache records:')
    display(test_skipped.head(20))

if WRITE_TEST:
    test_write = save_in_place(test_updated, TEST_PATH)
    print(f"WRITTEN: {test_write['path']}")
    print(f"rows: {test_write['rows']:,}")
    print(f"dialogues: {test_write['dialogues']:,}")
    print(f"file size: {test_write['size_mib']:.2f} MiB")
else:
    print(f'NOT WRITTEN: WRITE_TEST=False for {TEST_PATH.name}')

## 6. Interpreting an incremental write

A write updates only dialogues backed by usable cache in that run. Every other row is retained exactly as loaded, including annotations written by an earlier run. As new P11 files arrive, rerun the notebook and enable the relevant split's write flag. Invalid or mismatched files remain visible in the audit and can be rerun separately.